# 📈 Track B: Advanced Feature Engineering, Deep Learning & Trading Strategy
**Author:** Daf (Track B Specialist)  
**Universe:** 5 Liquid Bluechip IDX Stocks (`BBCA.JK`, `BBRI.JK`, `BMRI.JK`, `TLKM.JK`, `ASII.JK`)  
**Timeline:** 2018 – 2026  
**Core Objectives:**
1. **Advanced Feature Engineering:** 29 backward-looking technical indicators (Momentum, Trend, Lags, Volatility, Volume) strictly without lookahead bias.
2. **Walk-Forward Cross-Validation:** Time-series validation strictly executed on the Training Set (2018–2023).
3. **Model Development:** LightGBM Gradient Boosting & PyTorch Multi-layer LSTM Sequence Deep Learning.
4. **Signal Conversion & Trading Backtest:** Conversion of $P(\text{Up}) \in [0, 1]$ into `BUY`, `HOLD`, `SELL` signals and backtesting vs Buy & Hold benchmark.
5. **Database Ingestion:** Storing structured predictions in SQLite (`stock_prediction.db`).

---  
## 1. Setup & Environment Verification

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import lightgbm as lgb
import torch

# Ensure project root in path
sys.path.append(os.path.abspath('..'))

from src.data.preparation import split_time_series
from src.features.technical_indicators import extract_features, get_feature_columns
from src.models.validation import evaluate_predictions, run_walk_forward_cv
from src.models.lightgbm_model import LightGBMStockModel, lightgbm_fold_trainer
from src.models.lstm_model import LSTMStockTrainer
from src.models.ensemble_and_signals import generate_trading_signals, backtest_trading_strategy
from src.models.save_predictions import query_prediction_summary

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11

print(f"PyTorch: {torch.__version__} | LightGBM: {lgb.__version__} | Device: {'CUDA' if torch.cuda.is_available() else 'CPU'}")

---  
## 2. Load Data & Advanced Feature Engineering
All technical indicators are computed using split-adjusted OHLC per ticker to prevent inter-ticker leakage and look-ahead bias.

In [ ]:
data_path = '../data/processed/modeling_data_advanced.csv'
if not os.path.exists(data_path):
    baseline_df = pd.read_csv('../data/processed/modeling_data_baseline.csv')
    df = extract_features(baseline_df, drop_warmup=True)
    df.to_csv(data_path, index=False)
else:
    df = pd.read_csv(data_path)

df['date'] = pd.to_datetime(df['date'])
feature_cols = get_feature_columns()
print(f"Dataset Shape: {df.shape}")
print(f"Features ({len(feature_cols)}): {feature_cols}")
df[['ticker', 'date', 'close', 'adj_close', 'rsi_14', 'stoch_k', 'ema_crossover_5_20', 'target']].head()

### Strict Chronological Dataset Split
- **Train Set:** 2018-01-01 to 2023-12-31 (~7,153 rows)
- **Validation Set:** 2024-01-01 to 2024-12-31 (~1,185 rows) — Threshold & Ensemble Tuning
- **Test Set:** 2025-01-01 onwards (~1,950 rows) — ⚠️ 100% Sterile Evaluation

In [ ]:
train_df, val_df, test_df = split_time_series(df)
print(f"Train records: {len(train_df)} | Val records: {len(val_df)} | Test records: {len(test_df)}")

---  
## 3. Walk-Forward Cross-Validation (Train Set Only)
Expanding window cross-validation prevents forward leakage during model validation.

In [ ]:
cv_summary, fold_metrics = run_walk_forward_cv(
    train_df, 
    feature_cols,
    lightgbm_fold_trainer,
    n_splits=5,
    model_name='LightGBM'
)
display(cv_summary[['fold', 'val_start', 'val_end', 'roc_auc', 'accuracy', 'f1_score', 'brier_score']])

---  
## 4. Model 1: LightGBM Gradient Boosting Classifier

In [ ]:
lgbm_model = LightGBMStockModel()
lgbm_model.fit(
    train_df[feature_cols], train_df['target'],
    val_df[feature_cols], val_df['target'],
    early_stopping_rounds=30
)

# Feature Importance Visualization
top_features = lgbm_model.feature_importances_.head(15)
plt.figure(figsize=(10, 6))
sns.barplot(data=top_features, x='importance_gain', y='feature', palette='viridis')
plt.title('Top 15 Most Informative Features (LightGBM Gain Importance)', fontsize=14, fontweight='bold')
plt.xlabel('Importance Gain')
plt.ylabel('Feature')
plt.tight_layout()
plt.show()

---  
## 5. Model 2: PyTorch Deep Learning Sequence Model (LSTM)
Trained on sliding windows of $N=15$ consecutive trading days per ticker.

In [ ]:
lstm_trainer = LSTMStockTrainer(
    input_dim=len(feature_cols),
    seq_length=15,
    hidden_dim=64,
    num_layers=2,
    epochs=50,
    early_stopping_patience=10
)
lstm_trainer.fit(train_df, val_df, feature_cols)

# Plot Training and Validation Loss
plt.figure(figsize=(10, 5))
plt.plot(lstm_trainer.history['train_loss'], label='Train Loss (BCE)', color='#1f77b4', lw=2)
plt.plot(lstm_trainer.history['val_loss'], label='Val Loss (BCE)', color='#ff7f0e', lw=2)
plt.title('PyTorch LSTM Training Convergence & Early Stopping', fontsize=14, fontweight='bold')
plt.xlabel('Epoch')
plt.ylabel('Binary Cross-Entropy Loss')
plt.legend()
plt.tight_layout()
plt.show()

---  
## 6. Model Ensembling & Threshold Calibration (Validation Set: 2024)
We calibrate ensemble weights and trading signal thresholds solely on the Validation Set.

In [ ]:
val_lgb_prob = lgbm_model.predict_proba(val_df[feature_cols])
val_lstm_prob, val_meta = lstm_trainer.predict_proba(val_df, feature_cols)

val_merged = pd.merge(
    val_df.assign(prob_lgbm=val_lgb_prob),
    val_meta.assign(prob_lstm=val_lstm_prob)[['date', 'ticker', 'prob_lstm']],
    on=['date', 'ticker'],
    how='inner'
).sort_values(by=['date', 'ticker']).reset_index(drop=True)

# Search optimal ensemble weight
weight_grid = np.linspace(0.0, 1.0, 21)
auc_scores = []
for w in weight_grid:
    p = w * val_merged['prob_lgbm'] + (1.0 - w) * val_merged['prob_lstm']
    auc_scores.append(evaluate_predictions(val_merged['target'], p)['roc_auc'])

best_w = weight_grid[np.argmax(auc_scores)]
print(f"Optimal Validation Weight: LightGBM = {best_w:.2f}, LSTM = {1.0 - best_w:.2f} (Max Val AUC: {max(auc_scores):.4f})")

plt.figure(figsize=(8, 4))
plt.plot(weight_grid, auc_scores, marker='o', color='#2ca02c')
plt.axvline(best_w, color='red', linestyle='--', label=f'Best Weight = {best_w:.2f}')
plt.title('Validation ROC-AUC vs LightGBM Ensemble Weight', fontweight='bold')
plt.xlabel('LightGBM Weight ($w_1$)')
plt.ylabel('Validation ROC-AUC')
plt.legend()
plt.tight_layout()
plt.show()

---  
## 7. Final Out-of-Sample Evaluation & Trading Backtest (Test Set: 2025+)
Evaluating final model predictions on the completely unseen Test Set.

In [ ]:
test_lgb_prob = lgbm_model.predict_proba(test_df[feature_cols])
test_lstm_prob, test_meta = lstm_trainer.predict_proba(test_df, feature_cols)

test_merged = pd.merge(
    test_df.assign(prob_lgbm=test_lgb_prob),
    test_meta.assign(prob_lstm=test_lstm_prob)[['date', 'ticker', 'prob_lstm']],
    on=['date', 'ticker'],
    how='inner'
).sort_values(by=['date', 'ticker']).reset_index(drop=True)

test_merged['prob_ensemble'] = best_w * test_merged['prob_lgbm'] + (1.0 - best_w) * test_merged['prob_lstm']

# Generate Signals
for m_name in ['lgbm', 'lstm', 'ensemble']:
    pred, sig = generate_trading_signals(test_merged[f'prob_{m_name}'], buy_threshold=0.53, sell_threshold=0.47)
    test_merged[f'pred_{m_name}'] = pred
    test_merged[f'signal_{m_name}'] = sig

# Predictive Metrics Summary
m_lgb = evaluate_predictions(test_merged['target'], test_merged['prob_lgbm'])
m_lstm = evaluate_predictions(test_merged['target'], test_merged['prob_lstm'])
m_ens = evaluate_predictions(test_merged['target'], test_merged['prob_ensemble'])

comp_df = pd.DataFrame([
    {'Model': 'LightGBM', **m_lgb},
    {'Model': 'LSTM/GRU', **m_lstm},
    {'Model': 'Ensemble (Track B)', **m_ens}
])
display(comp_df[['Model', 'roc_auc', 'accuracy', 'f1_score', 'precision', 'recall', 'brier_score']])

### Realistic Trading Simulation vs Buy & Hold Benchmark

In [ ]:
bt_lgb = backtest_trading_strategy(test_merged, test_merged['signal_lgbm'])
bt_lstm = backtest_trading_strategy(test_merged, test_merged['signal_lstm'])
bt_ens = backtest_trading_strategy(test_merged, test_merged['signal_ensemble'])

bt_summary = pd.DataFrame([
    {'Strategy': 'Buy & Hold (Benchmark)', 'Total Return (%)': bt_lgb['benchmark_total_return']*100, 'Excess Return (%)': 0.0, 'Sharpe Ratio': bt_lgb['benchmark_sharpe'], 'Max Drawdown (%)': bt_lgb['benchmark_max_drawdown']*100, 'Trades': 0},
    {'Strategy': 'LightGBM Strategy', 'Total Return (%)': bt_lgb['strategy_total_return']*100, 'Excess Return (%)': bt_lgb['excess_return']*100, 'Sharpe Ratio': bt_lgb['strategy_sharpe'], 'Max Drawdown (%)': bt_lgb['strategy_max_drawdown']*100, 'Trades': bt_lgb['num_trades']},
    {'Strategy': 'LSTM / GRU Strategy', 'Total Return (%)': bt_lstm['strategy_total_return']*100, 'Excess Return (%)': bt_lstm['excess_return']*100, 'Sharpe Ratio': bt_lstm['strategy_sharpe'], 'Max Drawdown (%)': bt_lstm['strategy_max_drawdown']*100, 'Trades': bt_lstm['num_trades']},
    {'Strategy': 'Ensemble Strategy (Track B)', 'Total Return (%)': bt_ens['strategy_total_return']*100, 'Excess Return (%)': bt_ens['excess_return']*100, 'Sharpe Ratio': bt_ens['strategy_sharpe'], 'Max Drawdown (%)': bt_ens['strategy_max_drawdown']*100, 'Trades': bt_ens['num_trades']}
])
display(bt_summary)

---  
## 8. Database Records Verification
Verifying that predictions and signals have been committed to SQLite database (`stock_prediction.db`).

In [ ]:
db_summary = query_prediction_summary()
display(db_summary)